<a href="https://colab.research.google.com/github/fvangool/Deep-Learning-Specialization-Coursera/blob/main/Modles_ensembling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
Grandmaster Ensemble v4
========================
Changes vs v3:

  CRITICAL FIXES
  ──────────────
  + Fixed label mapping — uses Low=0, Medium=1, High=2 throughout
    (v3 used LabelEncoder which maps alphabetically: High=0, Low=1, Med=2
     making per-class diagnostics unreadable and confusing)
  + Stacker REMOVED — it was overfitting badly in v3:
      raw stacker OOF 0.97448 → bias +0.00529 → 0.98041 OOF
      but LB collapsed to 0.97846 (below v2's 0.98016)
      "Did not meet early stopping" on all 50 folds = noise fitting
  + Differential evolution ADDED BACK as post-hill-climb calibration
      Applied to hill-climb blend (not raw stacker)
      Narrower bounds (0.5, 2.5) instead of (0.2, 5.0) to prevent
      extreme weight collapse like [4.8, 0.3, 1.2]

  ENSEMBLE STRATEGY
  ─────────────────
  Method 1: Hill-climb weights + logit bias tuning
  Method 2: Hill-climb weights + differential evolution thresholds
  Method 3: Equal weights + logit bias tuning (sanity check)
  → All three submissions saved, submit the highest CV

  MODEL REGISTRY
  ──────────────
  + lgbm_baseline kept (single-seed Kaggle notebook — proven to work)
  + xgb_v22 (LB 0.97865 — best single XGB)
  + cat_v20 (new diverse model)
  + realmlp, autogluon (proven ensemble contributors in v3 hill-climb)
  - ftt, tabm, extratrees, gnn: got 0.000 weight in v3 hill-climb
    → disabled by default, can re-enable below

  KEY DIAGNOSTIC
  ──────────────
  Per-class BA now uses fixed Low=0/Med=1/High=2 mapping
  Hard-example BA reported pre and post calibration
"""

import numpy as np
import pandas as pd
import lightgbm as lgb
import os
import itertools
import warnings
warnings.filterwarnings("ignore")

from scipy.stats import rankdata
from scipy.special import logit
from scipy.optimize import differential_evolution
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
import wandb
from google.colab import userdata

# ============================================================
# SECTION 0 — CONFIGURATION
# ============================================================
OOF_DIR   = "/content/drive/MyDrive/irrigation_need_v15/"
SUB_DIR   = "/content/drive/MyDrive/irrigation_need_v15/"

HARD_IDX_PATH = f"{OOF_DIR}hard_example_indices.npy"

# ── W&B ──────────────────────────────────────────────────────
WANDB_PROJECT  = "ps-s6e4-irrigation"
WANDB_ENTITY   = "wblackstone-twilight-signals"
WANDB_ENABLED  = True
WB_TOKEN       = "WB_TOKEN"    # Colab secret key name
TAG            = "ensemble_v4"

# Fixed label mapping — matches XGB/CatBoost/LGBM v1 convention
TARGET_MAPPING     = {"Low": 0, "Medium": 1, "High": 2}
INV_TARGET_MAPPING = {0: "Low", 1: "Medium", 2: "High"}
CLASS_NAMES        = ["Low", "Medium", "High"]

# ── Model file registry ───────────────────────────────────────
# Models with 0.000 weight in v3 hill-climb are disabled.
# Re-enable by uncommenting — they won't hurt (weight collapses to 0)
# but slow down hill-climbing.
FILES = {
    # ── Core GBT models (proven contributors) ────────────────
    "xgb_v22"       : (f"{OOF_DIR}oof_xgb_v22_biased.npy",
                       f"{OOF_DIR}pred_xgb_v22_biased.npy"),
    "lgbm_baseline" : (f"{OOF_DIR}oof_lgbm_v1.npy",
                       f"{OOF_DIR}pred_lgbm_v1.npy"),
    "cat_v18"       : (f"{OOF_DIR}oof_cat_v18.npy",
                       f"{OOF_DIR}pred_cat_v18.npy"),
    # ── Neural / other (proven contributors) ─────────────────
    "realmlp"       : (f"{OOF_DIR}oof_realmlp_biased.npy",
                       f"{OOF_DIR}pred_realmlp_biased.npy"),
    "autogluon"     : (f"{OOF_DIR}oof_autogluon_v1.npy",
                       f"{OOF_DIR}pred_autogluon_v1.npy"),
    # ── Got 0.000 weight in v3 — disabled, uncomment to test ─
    # "ftt"          : (f"{OOF_DIR}oof_ftt_v2.npy",
    #                   f"{OOF_DIR}pred_ftt_v2.npy"),
    # "gnn"          : (f"{OOF_DIR}oof_gnn_v1.npy",
    #                   f"{OOF_DIR}pred_gnn_v1.npy"),
    # "tabm"         : (f"{OOF_DIR}oof_tabm_v5_biased.npy",
    #                   f"{OOF_DIR}pred_tabm_v5_biased.npy"),
    # "extratrees"   : (f"{OOF_DIR}oof_extratrees.npy",
    #                   f"{OOF_DIR}pred_extratrees.npy"),
}

# Differential evolution settings
# Narrower bounds prevent extreme class weight collapse
DIFFEVOL_BOUNDS    = (0.5, 2.5)   # was (0.2, 5.0) in v2 — too wide
DIFFEVOL_POPSIZE   = 20
DIFFEVOL_SEED      = 42


# ============================================================
# W&B HELPER
# ============================================================
class WandbLogger:
    """Thin wrapper — W&B failures never crash the ensemble."""
    def __init__(self, enabled=True):
        self.enabled = enabled
        self.run     = None

    def init(self, config):
        if not self.enabled:
            return
        try:
            api_key = userdata.get(WB_TOKEN)
            wandb.login(key=api_key, relogin=True)
            self.run = wandb.init(
                project = WANDB_PROJECT,
                entity  = WANDB_ENTITY,
                name    = TAG,
                config  = config,
                tags    = ["ensemble", "v4", "ps-s6e4"],
            )
            print(f"  W&B run: {self.run.url}")
        except Exception as e:
            print(f"  [W&B] init failed: {e}")
            self.enabled = False

    def log(self, metrics):
        if not self.enabled or self.run is None:
            return
        try:
            wandb.log(metrics)
        except Exception as e:
            print(f"  [W&B] log failed: {e}")

    def log_confusion_matrix(self, y_true, y_pred, title):
        if not self.enabled or self.run is None:
            return
        try:
            wandb.log({
                title: wandb.plot.confusion_matrix(
                    probs=None,
                    y_true=y_true.tolist(),
                    preds=y_pred.tolist(),
                    class_names=CLASS_NAMES,
                )
            })
        except Exception as e:
            print(f"  [W&B] confusion matrix failed: {e}")

    def log_artifact(self, local_path, artifact_name,
                     artifact_type, description=""):
        if not self.enabled or self.run is None:
            return
        try:
            art = wandb.Artifact(
                name=artifact_name, type=artifact_type,
                description=description,
            )
            art.add_file(local_path)
            self.run.log_artifact(art)
            print(f"  [W&B] artifact logged: {artifact_name}")
        except Exception as e:
            print(f"  [W&B] artifact failed: {e}")

    def summary(self, metrics):
        if not self.enabled or self.run is None:
            return
        try:
            for k, v in metrics.items():
                wandb.run.summary[k] = v
        except Exception as e:
            print(f"  [W&B] summary failed: {e}")

    def finish(self):
        if not self.enabled or self.run is None:
            return
        try:
            wandb.finish()
        except Exception as e:
            print(f"  [W&B] finish failed: {e}")


wb = WandbLogger(enabled=WANDB_ENABLED)

print("=" * 60)
print("  Grandmaster Ensemble v4")
print("=" * 60)

# ============================================================
# SECTION 1 — LOAD DATA
# ============================================================
train_df = pd.read_csv(f"{OOF_DIR}train.csv")
test_df  = pd.read_csv(f"{OOF_DIR}test.csv")

# Fixed mapping — do NOT use LabelEncoder (alphabetical order differs)
y_train    = train_df["Irrigation_Need"].map(TARGET_MAPPING).values
target_len = len(y_train)
test_ids   = test_df["id"].values

print(f"\n  Train rows  : {target_len:,}")
print(f"  Test rows   : {len(test_df):,}")
print(f"  Label map   : {TARGET_MAPPING}")
print(f"  Class dist  : "
      f"{dict(zip(*np.unique(y_train, return_counts=True)))}")

# Hard example indices
hard_mask = None
if os.path.exists(HARD_IDX_PATH):
    hard_idx  = np.load(HARD_IDX_PATH)
    hard_mask = np.zeros(target_len, dtype=bool)
    hard_mask[hard_idx] = True
    print(f"  Hard examples: {hard_mask.sum():,} rows loaded ✅")

# ── W&B init ─────────────────────────────────────────────────
wb.init(config={
    "tag"             : TAG,
    "version"         : "v4",
    "models"          : list(FILES.keys()),
    "diffevol_bounds" : DIFFEVOL_BOUNDS,
    "diffevol_popsize": DIFFEVOL_POPSIZE,
    "oof_dir"         : OOF_DIR,
})

# ============================================================
# SECTION 2 — LOAD OOF/PRED FILES
# ============================================================
print(f"\n[1] Loading OOF/pred files...")
raw_oofs, raw_preds, model_names = [], [], []

for name, (oof_p, pred_p) in FILES.items():
    if os.path.exists(oof_p) and os.path.exists(pred_p):
        oof  = np.load(oof_p)[:target_len]
        pred = np.load(pred_p)
        raw_oofs.append(oof)
        raw_preds.append(pred)
        model_names.append(name)
        print(f"  ✅ {name:<16} oof={oof.shape} pred={pred.shape}")
    else:
        print(f"  ⚠  {name:<16} NOT FOUND — skipping")

print(f"\n  Loaded: {len(model_names)} models")

# ============================================================
# SECTION 3 — ALIGNMENT (permutation search)
# ============================================================
print(f"\n[2] Aligning model probabilities...")
print(f"  {'Model':<16} {'Best BA':>10} {'Permutation':>14}  "
      f"{'Notes'}")
print("  " + "─" * 58)

aligned_oofs, aligned_preds = [], []
model_bas = {}

for i, name in enumerate(model_names):
    best_ba, best_perm = 0, (0, 1, 2)
    for p in itertools.permutations([0, 1, 2]):
        score = balanced_accuracy_score(
            y_train, raw_oofs[i][:, p].argmax(axis=1)
        )
        if score > best_ba:
            best_ba, best_perm = score, p

    aligned_oofs.append(raw_oofs[i][:, best_perm])
    aligned_preds.append(raw_preds[i][:, best_perm])
    model_bas[name] = best_ba

    note = ""
    if best_perm != (0, 1, 2):
        note = f"⚠ remapped {best_perm}"
    print(f"  {name:<16} {best_ba:>10.5f} {str(best_perm):>14}  {note}")

# Log individual model BAs to W&B
wb.log({f"ba_individual_{n}": v for n, v in model_bas.items()})

# ============================================================
# SECTION 4 — HILL-CLIMB WEIGHT SEARCH
# ============================================================
print(f"\n[3] Hill-climb weight search...")

n_models        = len(aligned_oofs)
best_weights_hc = np.ones(n_models) / n_models
best_ba_hc      = balanced_accuracy_score(
    y_train,
    np.average(aligned_oofs, axis=0,
               weights=best_weights_hc).argmax(1),
)

for step in [0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001]:
    improved = True
    while improved:
        improved = False
        for i in range(n_models):
            for delta in [step, -step]:
                trial    = best_weights_hc.copy()
                trial[i] = max(0, trial[i] + delta)
                if trial.sum() == 0:
                    continue
                trial = trial / trial.sum()
                s = balanced_accuracy_score(
                    y_train,
                    np.average(aligned_oofs, axis=0,
                               weights=trial).argmax(1),
                )
                if s > best_ba_hc + 1e-9:
                    best_ba_hc    = s
                    best_weights_hc = trial
                    improved      = True

print(f"  Hill-climb OOF BA: {best_ba_hc:.6f}")
print(f"\n  {'Model':<16} {'Weight':>8} {'Indiv BA':>10}")
print("  " + "─" * 40)
for name, w in zip(model_names, best_weights_hc):
    bar    = "█" * int(w * 40)
    status = "  " + bar if w > 0 else "  (excluded)"
    print(f"  {name:<16} {w:>8.4f} "
          f"{model_bas[name]:>10.5f}{status}")

# Log hill-climb weights and OOF BA to W&B
wb.log({
    "ba_hillclimb_raw": best_ba_hc,
    **{f"weight_{n}": float(w)
       for n, w in zip(model_names, best_weights_hc)},
})

# Hill-climb blended probabilities
hc_oof_blend  = np.average(
    aligned_oofs, axis=0, weights=best_weights_hc
).astype(np.float32)
hc_test_blend = np.average(
    aligned_preds, axis=0, weights=best_weights_hc
).astype(np.float32)

# ============================================================
# SECTION 5 — METHOD 1: HILL-CLIMB + LOGIT BIAS
# ============================================================
print(f"\n[4] Method 1: Hill-climb + logit-space bias tuning...")

def apply_bias(probs, bias):
    log_p = logit(np.clip(probs, 1e-15, 1-1e-15)) + bias
    exp_p = np.exp(log_p)
    return (exp_p / exp_p.sum(axis=1, keepdims=True)).astype(np.float32)

def tune_logit_bias(oof_probs, y_true):
    def get_preds(probs, bias):
        adj = logit(np.clip(probs, 1e-15, 1-1e-15)) + bias
        return np.argmax(adj, axis=1)

    best_bias   = np.zeros(3)
    best_score  = balanced_accuracy_score(y_true, oof_probs.argmax(1))
    raw_score   = best_score

    for step in [1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002]:
        improved = True
        while improved:
            improved = False
            for ci in range(3):
                for d in [1, -1]:
                    trial      = best_bias.copy()
                    trial[ci] += d * step
                    s = balanced_accuracy_score(
                        y_true, get_preds(oof_probs, trial)
                    )
                    if s > best_score + 1e-9:
                        best_score, best_bias, improved = s, trial, True

    print(f"  Bias raw  : {raw_score:.6f}")
    print(f"  Bias tuned: {best_score:.6f} "
          f"(+{best_score - raw_score:.6f})")
    print(f"  Biases    : Low={best_bias[0]:.4f} "
          f"Med={best_bias[1]:.4f} High={best_bias[2]:.4f}")
    return best_bias, best_score

bias_hc, ba_method1 = tune_logit_bias(hc_oof_blend, y_train)
m1_oof_cal  = apply_bias(hc_oof_blend,  bias_hc)
m1_test_cal = apply_bias(hc_test_blend, bias_hc)

print(f"\n  Per-class OOF BA (Method 1 — Low=0, Med=1, High=2):")
m1_per_class = {}
for cls in range(3):
    mask = y_train == cls
    ba   = (m1_oof_cal[mask].argmax(1) == cls).mean()
    m1_per_class[f"ba_m1_{CLASS_NAMES[cls].lower()}"] = float(ba)
    print(f"    {CLASS_NAMES[cls]:<8}: {ba:.5f}")

wb.log({
    "ba_m1_logit_bias": ba_method1,
    "bias_m1_low"     : float(bias_hc[0]),
    "bias_m1_medium"  : float(bias_hc[1]),
    "bias_m1_high"    : float(bias_hc[2]),
    **m1_per_class,
})
wb.log_confusion_matrix(
    y_true=y_train,
    y_pred=m1_oof_cal.argmax(axis=1),
    title="m1_oof_confusion_matrix",
)

if hard_mask is not None:
    hba = balanced_accuracy_score(
        y_train[hard_mask], m1_oof_cal[hard_mask].argmax(1)
    )
    print(f"  Hard-example BA: {hba:.5f}")
    wb.log({"ba_m1_hard_examples": float(hba)})

# ============================================================
# SECTION 6 — METHOD 2: HILL-CLIMB + DIFFERENTIAL EVOLUTION
# ============================================================
print(f"\n[5] Method 2: Hill-climb + differential evolution thresholds...")
print(f"  Bounds: {DIFFEVOL_BOUNDS}  "
      f"(narrower than v2 to prevent extreme weights)")

def neg_ba_weights(weights, probs, y_true):
    adj = probs * np.array(weights)
    adj = adj / adj.sum(axis=1, keepdims=True)
    return -balanced_accuracy_score(y_true, adj.argmax(axis=1))

de_result = differential_evolution(
    neg_ba_weights,
    args=(hc_oof_blend, y_train),
    bounds=[DIFFEVOL_BOUNDS] * 3,
    seed=DIFFEVOL_SEED,
    popsize=DIFFEVOL_POPSIZE,
    maxiter=1000,
    tol=1e-7,
    disp=False,
)
de_weights  = de_result.x
ba_method2  = -de_result.fun

print(f"  DE weights: Low={de_weights[0]:.4f} "
      f"Med={de_weights[1]:.4f} High={de_weights[2]:.4f}")
print(f"  Method 2 OOF BA: {ba_method2:.6f}")

# Apply DE weights to OOF and test
m2_oof_cal  = (hc_oof_blend  * de_weights)
m2_oof_cal  = (m2_oof_cal / m2_oof_cal.sum(axis=1, keepdims=True)
               ).astype(np.float32)
m2_test_cal = (hc_test_blend * de_weights)
m2_test_cal = (m2_test_cal / m2_test_cal.sum(axis=1, keepdims=True)
               ).astype(np.float32)

print(f"\n  Per-class OOF BA (Method 2):")
m2_per_class = {}
for cls in range(3):
    mask = y_train == cls
    ba   = (m2_oof_cal[mask].argmax(1) == cls).mean()
    m2_per_class[f"ba_m2_{CLASS_NAMES[cls].lower()}"] = float(ba)
    print(f"    {CLASS_NAMES[cls]:<8}: {ba:.5f}")

wb.log({
    "ba_m2_diffevol"  : ba_method2,
    "de_weight_low"   : float(de_weights[0]),
    "de_weight_medium": float(de_weights[1]),
    "de_weight_high"  : float(de_weights[2]),
    **m2_per_class,
})
wb.log_confusion_matrix(
    y_true=y_train,
    y_pred=m2_oof_cal.argmax(axis=1),
    title="m2_oof_confusion_matrix",
)

if hard_mask is not None:
    hba2 = balanced_accuracy_score(
        y_train[hard_mask], m2_oof_cal[hard_mask].argmax(1)
    )
    print(f"  Hard-example BA: {hba2:.5f}")
    wb.log({"ba_m2_hard_examples": float(hba2)})

# ============================================================
# SECTION 7 — METHOD 3: EQUAL WEIGHTS + LOGIT BIAS (sanity)
# ============================================================
print(f"\n[6] Method 3: Equal weights + logit bias (sanity check)...")

eq_oof_blend  = np.mean(aligned_oofs,  axis=0).astype(np.float32)
eq_test_blend = np.mean(aligned_preds, axis=0).astype(np.float32)

bias_eq, ba_method3 = tune_logit_bias(eq_oof_blend, y_train)
m3_oof_cal  = apply_bias(eq_oof_blend,  bias_eq)
m3_test_cal = apply_bias(eq_test_blend, bias_eq)

print(f"  Equal weights + bias OOF BA: {ba_method3:.6f}")
wb.log({"ba_m3_equal_bias": ba_method3})

# ============================================================
# SECTION 8 — COMPARISON + SUBMISSION
# ============================================================
# ── W&B summary ──────────────────────────────────────────────
wb.summary({
    "ba_hillclimb_raw"  : best_ba_hc,
    "ba_m1_logit_bias"  : ba_method1,
    "ba_m2_diffevol"    : ba_method2,
    "ba_m3_equal_bias"  : ba_method3,
    "best_method"       : max(
        [("m1", ba_method1), ("m2", ba_method2), ("m3", ba_method3)],
        key=lambda x: x[1]
    )[0],
    "n_models"          : len(model_names),
    "models_used"       : str(model_names),
})

print(f"\n{'='*60}")
print(f"RESULTS COMPARISON")
print(f"{'='*60}")
print(f"  {'Method':<45} {'OOF BA':>10}")
print(f"  {'─'*56}")

for name, ba in sorted(
    model_bas.items(), key=lambda x: -x[1]
):
    print(f"  {'Individual: '+name:<45} {ba:>10.5f}")

print(f"  {'─'*56}")
print(f"  {'Hill-climb (raw)':<45} {best_ba_hc:>10.5f}")
print(f"  {'Method 1: Hill-climb + logit bias':<45} {ba_method1:>10.5f}")
print(f"  {'Method 2: Hill-climb + diff evolution':<45} {ba_method2:>10.5f}")
print(f"  {'Method 3: Equal weights + logit bias':<45} {ba_method3:>10.5f}")
print(f"{'='*60}")

methods = [
    ("m1_logit_bias",    m1_test_cal, ba_method1),
    ("m2_diffevol",      m2_test_cal, ba_method2),
    ("m3_equal_bias",    m3_test_cal, ba_method3),
]

best_method_name = max(methods, key=lambda x: x[2])[0]

for method_name, test_probs, cv_score in methods:
    preds = test_probs.argmax(axis=1)
    labels = [INV_TARGET_MAPPING[p] for p in preds]
    sub = pd.DataFrame({
        "id"              : test_ids,
        "Irrigation_Need" : labels,
    })
    out_path = f"{SUB_DIR}ensemble_v4_{method_name}.csv"
    sub.to_csv(out_path, index=False)
    marker = " ← highest CV" if method_name == best_method_name else ""
    print(f"\n  Saved: ensemble_v4_{method_name}.csv "
          f"(CV={cv_score:.6f}){marker}")
    print(sub["Irrigation_Need"].value_counts().to_string())
    wb.log_artifact(
        local_path    = out_path,
        artifact_name = f"submission_ensemble_v4_{method_name}",
        artifact_type = "submission",
        description   = f"Ensemble v4 {method_name} | CV={cv_score:.6f}",
    )
    wb.log({
        f"submission_cv_{method_name}": cv_score,
        f"submission_n_low_{method_name}": int(
            (sub['Irrigation_Need']=='Low').sum()
        ),
        f"submission_n_medium_{method_name}": int(
            (sub['Irrigation_Need']=='Medium').sum()
        ),
        f"submission_n_high_{method_name}": int(
            (sub['Irrigation_Need']=='High').sum()
        ),
    })

print(f"\n{'='*60}")
print(f"  Recommended: ensemble_v4_{best_method_name}.csv")
print(f"  Also worth submitting: ensemble_v4_m1_logit_bias.csv")
print(f"  (logit bias is more stable — may generalise better to LB)")
print(f"{'='*60}")

wb.finish()

  Grandmaster Ensemble v4

  Train rows  : 630,000
  Test rows   : 270,000
  Label map   : {'Low': 0, 'Medium': 1, 'High': 2}
  Class dist  : {np.int64(0): np.int64(369917), np.int64(1): np.int64(239074), np.int64(2): np.int64(21009)}
  Hard examples: 7,377 rows loaded ✅


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


ba_individual_autogluon,▁
ba_individual_lgbm_baseline,▁
ba_individual_realmlp,▁
ba_individual_xgb_v22,▁
ba_individual_autogluon,0.96773
ba_individual_lgbm_baseline,0.97528
ba_individual_realmlp,0.97807
ba_individual_xgb_v22,0.97941


  W&B run: https://wandb.ai/wblackstone-twilight-signals/ps-s6e4-irrigation/runs/sc2evde4

[1] Loading OOF/pred files...
  ✅ xgb_v22          oof=(630000, 3) pred=(270000, 3)
  ✅ lgbm_baseline    oof=(630000, 3) pred=(270000, 3)
  ✅ cat_v18          oof=(630000, 3) pred=(270000, 3)
  ✅ realmlp          oof=(630000, 3) pred=(270000, 3)
  ✅ autogluon        oof=(630000, 3) pred=(270000, 3)

  Loaded: 5 models

[2] Aligning model probabilities...
  Model               Best BA    Permutation  Notes
  ──────────────────────────────────────────────────────────
  xgb_v22             0.97941      (0, 1, 2)  
  lgbm_baseline       0.97528      (0, 1, 2)  
  cat_v18             0.97935      (0, 1, 2)  
  realmlp             0.97807      (0, 1, 2)  
  autogluon           0.96773      (1, 2, 0)  ⚠ remapped (1, 2, 0)

[3] Hill-climb weight search...
  Hill-climb OOF BA: 0.980002

  Model              Weight   Indiv BA
  ────────────────────────────────────────
  xgb_v22            0.2570    0.97941